# 第三階段：排程 / 預測 / 模擬分析

本 Notebook 直接從 Google Sheets 抓取研究室伺服器的真實監控資料，進行：
1. **排程建議** — 熱力圖分析最佳使用時段
2. **預測** — 加權移動平均預測未來登入趨勢
3. **模擬** — What-if 情境模擬資源使用率

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from ipywidgets import interact, IntSlider, Dropdown
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.facecolor'] = '#0b0e14'
matplotlib.rcParams['axes.facecolor'] = '#131720'
matplotlib.rcParams['text.color'] = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor'] = '#94a3b8'
matplotlib.rcParams['xtick.color'] = '#64748b'
matplotlib.rcParams['ytick.color'] = '#64748b'

## 從 Google Sheets 載入真實資料

In [ ]:
SHEET_ID = '1QMGRYLVxgGKr7JMq6phXEdmllzYyOuVUK4VEY30YLHQ'

url_ssh = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet=工作表1'
url_container = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet=容器進入紀錄'

df_ssh = pd.read_csv(url_ssh)
df_container = pd.read_csv(url_container)

print(f'SSH 登入紀錄: {len(df_ssh)} 筆')
print(f'容器進入紀錄: {len(df_container)} 筆')
print()
print('=== SSH 登入 (前 5 筆) ===')
display(df_ssh.head())
print('\n=== 容器進入 (前 5 筆) ===')
display(df_container.head())

In [ ]:
# 統一時間欄位解析
def parse_time(s):
    s = str(s).strip()
    # 處理 "2026/3/25 上午 2:15:51" 格式
    for am_str in ['上午', 'AM']:
        if am_str in s:
            s = s.replace(am_str, '').strip()
            try:
                return pd.to_datetime(s, format='%Y/%m/%d %H:%M:%S')
            except:
                pass
    for pm_str in ['下午', 'PM']:
        if pm_str in s:
            s = s.replace(pm_str, '').strip()
            try:
                dt = pd.to_datetime(s, format='%Y/%m/%d %H:%M:%S')
                if dt.hour < 12:
                    dt = dt + timedelta(hours=12)
                return dt
            except:
                pass
    # 處理 "2026-03-25 2:57:07" 格式
    try:
        return pd.to_datetime(s)
    except:
        return pd.NaT

time_col_ssh = df_ssh.columns[0]
time_col_container = df_container.columns[0]

df_ssh['parsed_time'] = df_ssh[time_col_ssh].apply(parse_time)
df_container['parsed_time'] = df_container[time_col_container].apply(parse_time)

df_ssh = df_ssh.dropna(subset=['parsed_time'])
df_container = df_container.dropna(subset=['parsed_time'])

# 合併所有登入紀錄
df_ssh['type'] = 'SSH'
df_container['type'] = '容器'

user_col_ssh = df_ssh.columns[1]
user_col_container = df_container.columns[1]

df_all = pd.concat([
    df_ssh[['parsed_time', user_col_ssh, 'type']].rename(columns={user_col_ssh: 'user'}),
    df_container[['parsed_time', user_col_container, 'type']].rename(columns={user_col_container: 'user'}),
], ignore_index=True).sort_values('parsed_time', ascending=False)

df_all['hour'] = df_all['parsed_time'].dt.hour
df_all['date'] = df_all['parsed_time'].dt.date
df_all['weekday'] = df_all['parsed_time'].dt.weekday  # 0=Monday

print(f'合併後總紀錄: {len(df_all)} 筆')
print(f'時間範圍: {df_all["parsed_time"].min()} ~ {df_all["parsed_time"].max()}')
print(f'\n使用者分布:')
print(df_all['user'].value_counts().to_string())

## 計算統計數據

In [ ]:
# 24 小時登入分布
hour_stats = np.zeros(24)
hour_counts = df_all['hour'].value_counts()
for h, count in hour_counts.items():
    hour_stats[int(h)] = count

# 過去 7 天每日登入次數
today = df_all['parsed_time'].max().date()
last_7_days = [today - timedelta(days=i) for i in range(6, -1, -1)]
daily_counts = df_all.groupby('date').size()

week_labels = []
week_data = []
day_names = ['週一', '週二', '週三', '週四', '週五', '週六', '週日']
for d in last_7_days:
    count = daily_counts.get(d, 0)
    week_data.append(count)
    week_labels.append(f'{day_names[d.weekday()]}\n{d.month}/{d.day}')

week_data = np.array(week_data)

# 容器使用統計（從容器紀錄中提取各容器的使用頻率）
container_users = df_container[user_col_container].value_counts()

print('24 小時分布:', hour_stats.astype(int))
print(f'\n過去 7 天每日登入: {dict(zip([d.strftime("%m/%d") for d in last_7_days], week_data))}')
print(f'\n容器使用次數:\n{container_users.to_string()}')

---
## 1. 排程建議 — 最佳使用時段熱力圖

將每日登入量與小時分布交叉，產生 7×24 的負載矩陣，用熱力圖呈現。
顏色越深代表該時段使用量越高，綠色區域為建議使用的低負載時段。

In [ ]:
# 產生真實的 7x24 熱力圖矩陣
heatmap = np.zeros((7, 24))
for _, row in df_all.iterrows():
    wd = row['weekday']
    hr = row['hour']
    heatmap[wd, hr] += 1

# 繪製熱力圖
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap, aspect='auto', cmap='RdYlGn_r', interpolation='nearest')

ax.set_yticks(range(7))
ax.set_yticklabels(day_names)
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h}:00' for h in range(0, 24, 2)])
ax.set_xlabel('時間')
ax.set_title('伺服器使用負載熱力圖（綠色 = 低負載，紅色 = 高負載）', fontsize=13, pad=12)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('登入次數', color='#94a3b8')
cbar.ax.yaxis.set_tick_params(color='#64748b')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='#64748b')

plt.tight_layout()
plt.show()

# 找出最佳使用時段（排除深夜 0-6 點）
best_slots = []
for d in range(7):
    for h in range(7, 23):
        best_slots.append((heatmap[d, h], d, h))
best_slots.sort()

print('\n🟢 建議使用時段（負載最低的時段）：')
seen = set()
count = 0
for val, d, h in best_slots:
    key = (d, h // 3)
    if key in seen:
        continue
    seen.add(key)
    print(f'   {day_names[d]} {h:02d}:00 ~ {h+2:02d}:00 （登入次數: {val:.0f}）')
    count += 1
    if count >= 5:
        break

---
## 2. 預測 — 未來 3 天登入趨勢

使用**加權移動平均 (Weighted Moving Average)** 進行預測：
- 越近的日子權重越高（近 3 天佔 70%）
- 預測值會遞迴地作為下一天的輸入

In [ ]:
# 加權移動平均預測
weights = np.array([0.05, 0.05, 0.1, 0.1, 0.15, 0.25, 0.3])

working = list(week_data.astype(float))
predicted = []
for i in range(3):
    val = round(np.dot(working[-7:], weights))
    predicted.append(val)
    working.append(val)

pred_labels = []
for i in range(1, 4):
    d = today + timedelta(days=i)
    pred_labels.append(f'{day_names[d.weekday()]}\n{d.month}/{d.day}')

# 繪製趨勢圖
fig, ax = plt.subplots(figsize=(12, 5))

x_real = np.arange(7)
x_pred = np.arange(6, 10)
pred_with_bridge = [week_data[-1]] + predicted

ax.plot(x_real, week_data, 'o-', color='#00e87a', linewidth=2, markersize=6, label='歷史資料', zorder=3)
ax.fill_between(x_real, week_data, alpha=0.1, color='#00e87a')

ax.plot(x_pred, pred_with_bridge, 's--', color='#3b82f6', linewidth=2, markersize=6, label='預測值', zorder=3)
ax.fill_between(x_pred, pred_with_bridge, alpha=0.1, color='#3b82f6')

# 信賴區間
std = np.std(week_data) * 0.3 if np.std(week_data) > 0 else 1
upper = np.array(pred_with_bridge) + np.array([0, std, std*1.5, std*2])
lower = np.array(pred_with_bridge) - np.array([0, std, std*1.5, std*2])
lower = np.maximum(lower, 0)
ax.fill_between(x_pred, lower, upper, alpha=0.08, color='#3b82f6')

all_labels = week_labels + pred_labels
ax.set_xticks(range(10))
ax.set_xticklabels(all_labels, fontsize=9)
ax.set_ylabel('登入次數')
ax.set_title('7 天歷史趨勢 + 未來 3 天預測', fontsize=13, pad=12)
ax.legend(facecolor='#1a2030', edgecolor='#ffffff12', labelcolor='#e2e8f0')
ax.grid(True, alpha=0.1)
ax.axvline(x=6.5, color='#64748b', linestyle=':', alpha=0.5)
ax.text(6.5, max(max(week_data), max(predicted)) * 1.05, '← 歷史 | 預測 →', ha='center', fontsize=9, color='#64748b')

plt.tight_layout()
plt.show()

# 預測摘要
print('\n📊 預測結果：')
last = int(week_data[-1])
for i, val in enumerate(predicted):
    diff = val - last
    arrow = '↑' if diff > 0 else '↓' if diff < 0 else '→'
    sign = '+' if diff > 0 else ''
    d = today + timedelta(days=i+1)
    print(f'   {d.strftime("%m/%d")} ({day_names[d.weekday()]}): {val} 次  {arrow} ({sign}{diff})')
    last = val

trend = predicted[-1] - int(week_data[-1])
if trend > 2:
    print(f'\n⚠️  整體趨勢上升，預計未來使用量增加')
elif trend < -2:
    print(f'\n✅ 整體趨勢下降，伺服器壓力將減輕')
else:
    print(f'\n➡️  整體趨勢持平，維持目前水準')

---
## 3. What-if 資源模擬

根據歷史使用資料，模擬不同使用者數量和容器配置下的系統負載。

**模型假設（基於真實觀測資料）：**
- 從容器進入紀錄推算各容器的活躍程度
- 每位使用者平均佔用的資源比例由歷史使用頻率推算
- GPU 強度影響整體 GPU 佔用倍率

In [ ]:
# 從真實資料計算模擬參數
total_records = len(df_all)
unique_users = df_all['user'].nunique()
unique_containers = df_container[user_col_container].nunique()

# 每小時平均活躍人數（用來推算資源佔用）
active_hours = df_all.groupby([df_all['parsed_time'].dt.date, 'hour']).size()
avg_concurrent = active_hours.mean() if len(active_hours) > 0 else 1

# 尖峰時段（9-18點）的平均同時使用人數
peak_mask = (df_all['hour'] >= 9) & (df_all['hour'] <= 18)
peak_records = df_all[peak_mask]
peak_hourly = peak_records.groupby([peak_records['parsed_time'].dt.date, 'hour']).size()
avg_peak_concurrent = peak_hourly.mean() if len(peak_hourly) > 0 else 1

# 基準資源估計（每位同時在線使用者的平均資源佔用）
base_cpu_per_user = min(100 / max(avg_peak_concurrent, 1) * 0.6, 25)
base_mem_per_user = min(100 / max(avg_peak_concurrent, 1) * 0.5, 20)
base_gpu_per_user = min(100 / max(avg_peak_concurrent, 1) * 0.7, 30)

print(f'=== 模擬基準參數（從真實資料推算） ===')
print(f'歷史總紀錄數: {total_records}')
print(f'不重複使用者: {unique_users} 人')
print(f'不重複容器: {unique_containers} 個')
print(f'每時段平均活躍紀錄: {avg_concurrent:.1f}')
print(f'尖峰時段平均紀錄: {avg_peak_concurrent:.1f}')
print(f'\n推算每人平均佔用:')
print(f'  CPU: {base_cpu_per_user:.1f}%  |  MEM: {base_mem_per_user:.1f}%  |  GPU: {base_gpu_per_user:.1f}%')

In [ ]:
def simulate(users=3, active_containers=4, gpu_intensity='中'):
    gpu_mult = {'低': 0.4, '中': 1.0, '高': 1.8}[gpu_intensity]
    
    est_cpu = min(base_cpu_per_user * users + active_containers * 3, 100)
    est_mem = min(base_mem_per_user * users + active_containers * 4, 100)
    est_gpu = min(base_gpu_per_user * users * gpu_mult, 100)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    metrics = [
        ('CPU', est_cpu),
        ('MEMORY', est_mem),
        ('GPU', est_gpu),
    ]
    
    for ax, (label, val) in zip(axes, metrics):
        gauge_color = '#00e87a' if val <= 60 else '#f59e0b' if val <= 85 else '#ef4444'
        sizes = [val, 100 - val]
        colors_pie = [gauge_color, '#1a2030']
        
        wedges, _ = ax.pie(
            sizes, colors=colors_pie, startangle=90,
            wedgeprops={'width': 0.3, 'edgecolor': '#0b0e14', 'linewidth': 2}
        )
        
        ax.text(0, 0, f'{val:.0f}%', ha='center', va='center',
                fontsize=22, fontweight='bold', color=gauge_color)
        ax.set_title(label, fontsize=11, color='#94a3b8', pad=8)
    
    fig.suptitle(
        f'模擬配置：{users} 位使用者 / {active_containers} 個容器 / GPU強度: {gpu_intensity}',
        fontsize=11, color='#64748b', y=1.02
    )
    plt.tight_layout()
    plt.show()
    
    max_load = max(est_cpu, est_mem, est_gpu)
    if max_load <= 60:
        print(f'\n✅ 系統負載正常 ({max_load:.0f}%)，可安全增加使用者或容器')
    elif max_load <= 85:
        print(f'\n⚠️  系統負載中等 ({max_load:.0f}%)，建議謹慎增加負載')
    else:
        print(f'\n🔴 系統負載過高 ({max_load:.0f}%)，不建議增加更多使用者')
    
    print(f'    CPU: {est_cpu:.1f}%  |  MEM: {est_mem:.1f}%  |  GPU: {est_gpu:.1f}%')

interact(
    simulate,
    users=IntSlider(min=1, max=20, value=int(avg_peak_concurrent), description='在線人數'),
    active_containers=IntSlider(min=1, max=max(unique_containers, 8), value=unique_containers, description='容器數'),
    gpu_intensity=Dropdown(options=['低', '中', '高'], value='中', description='GPU強度')
)

---
## 總結

| 功能 | 方法 | 資料來源 | 輸出 |
|------|------|----------|------|
| 排程建議 | 星期×小時 交叉統計 | Google Sheets 真實登入紀錄 | 熱力圖 + 建議時段 |
| 趨勢預測 | 加權移動平均 (WMA) | 過去 7 天真實登入數 | 未來 3 天預測值 + 信賴區間 |
| 資源模擬 | What-if 參數模擬 | 真實使用頻率推算基準值 | CPU/MEM/GPU 預估 + 安全評估 |

這三個分析功能搭配第一、二階段的即時監控 Dashboard，形成完整的 MES 流程：

**資料收集 → 即時監控 → 排程優化 → 預測分析 → 情境模擬**